## LangChain Fundamentals

LangChain provides clean abstractions for building LLM applications.
Instead of writing boilerplate API calls, LangChain gives you 
reusable components that connect together cleanly.

Topics:
1. Prompt Templates — reusable parameterized prompts
2. LangChain + Groq setup
3. Output Parsers — structured extraction automatically
4. Chains — connecting components in sequence
5. Project: Multi-step fraud investigation chain

In [2]:
# ============================================================
# CELL 1: LangChain Setup
# ============================================================

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize Langchain's Groq client
llm  = ChatGroq(
    model = "llama-3.1-8b-instant",
    temperature = 0.0,
    max_tokens = 300,
    api_key=os.environ.get("GROQ_API_KEY")
)

# Test basic call
response = llm.invoke("What are the top 3 fraud indicators in banking")
print(response.content)
print(f"\nToken usage: {response.usage_metadata}")

Based on industry research and trends, here are the top 3 fraud indicators in banking:

1. **Unusual or Unexplained Transactions**: This includes transactions that are significantly higher or lower than the customer's average transaction amount, transactions that occur in a different location or time zone, or transactions that are made using a new or unfamiliar payment method. For example, a customer who typically makes small purchases online may suddenly make a large cash withdrawal from an ATM in a different city.

2. **Changes in Customer Behavior**: This includes changes in a customer's behavior, such as an increase in the frequency or amount of transactions, or a change in the types of transactions being made. For example, a customer who typically only uses their debit card for small purchases may suddenly start using it to make large purchases or to withdraw cash from ATMs.

3. **Inconsistencies in Customer Information**: This includes inconsistencies in a customer's name, addres

In [6]:
# ============================================================
# CELL 2: Prompt Templates
# ============================================================

# --- Basic Prompt Template ---
# Define a reusable template with variables in {curly braces}

fraud_template = ChatPromptTemplate.from_messages([
    ("system", """You are a senior fraud analyst at a bank.
Analyze the transaction and provide a concise risk assessment.
Keep your response under 100 words."""),
    ("user", """Transaction Details:
- Amount: ${amount}
- Merchant: {merchant}
- Time: {time}
- Customer avg spend: ${avg_spend}
- Location: {location}

Provide risk level (HIGH/MEDIUM/LOW) and key reason.""")
])

# Fill in the template with actual values
prompt_filled = fraud_template.invoke({
    "amount": "4500",
    "merchant": "Electronics Store",
    "time": "3:15 AM",
    "avg_spend": "85",
    "location": "Different country"
})

print("=== Filled Prompt (what gets sent to LLM) ===")
print(prompt_filled)

# Send filled prompt to LLM
response = llm.invoke(prompt_filled)
print("\n=== LLM Response ===")
print(response.content)

=== Filled Prompt (what gets sent to LLM) ===
messages=[SystemMessage(content='You are a senior fraud analyst at a bank.\nAnalyze the transaction and provide a concise risk assessment.\nKeep your response under 100 words.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Transaction Details:\n- Amount: $4500\n- Merchant: Electronics Store\n- Time: 3:15 AM\n- Customer avg spend: $85\n- Location: Different country\n\nProvide risk level (HIGH/MEDIUM/LOW) and key reason.', additional_kwargs={}, response_metadata={})]

=== LLM Response ===
Risk Level: HIGH

Key Reason: The transaction occurs at 3:15 AM, which is an unusual time for a legitimate purchase. Additionally, the amount ($4500) is significantly higher than the customer's average spend ($85), and the transaction is made from a different country, which may indicate international money laundering or card-not-present (CNP) fraud.


In [9]:
# ============================================================
# CELL 3: LCEL — LangChain Expression Language
# The pipe operator that chains components together
# ============================================================

from langchain_core.output_parsers import StrOutputParser

# LCEL uses the | (pipe) operator to chain components
# prompt | llm | parser
# Each component's output feeds into the next

# Build a chain
fraud_chain = fraud_template | llm | StrOutputParser()

# Now call the entire chain with one invoke
result = str(fraud_chain.invoke({
    "amount": "9500",
    "merchant": "Wire Transfer",
    "time": "2:30 AM",
    "avg_spend": "120",
    "location": "Nigeria"
}))

print("=== Chain Result ===")
print(result)
print(f"\nType: {type(result)}")

=== Chain Result ===
Risk Level: HIGH

Key Reason: The transaction amount ($9500) is significantly higher than the customer's average spend ($120), indicating a potential anomaly. Additionally, the merchant type (Wire Transfer) and location (Nigeria) are often associated with high-risk transactions, particularly at an unusual time (2:30 AM). These factors suggest a potential fraudulent activity, warranting further investigation.

Type: <class 'str'>


In [10]:
# ============================================================
# CELL 4: JSON Output Parser
# Replaces manual json.loads() with automatic structured parsing
# ============================================================

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Define the JSON structure you want
json_template = ChatPromptTemplate.from_messages([
    ("system", """You are a fraud detection system.
Analyze the transaction and respond ONLY with a JSON object.
No markdown, no explanation, pure JSON only.

Required format:
{{
    "risk_level": "HIGH" or "MEDIUM" or "LOW",
    "fraud_probability": float between 0.0 and 1.0,
    "red_flags": ["list", "of", "flags"],
    "recommended_action": "string",
    "requires_human_review": true or false
}}"""),
    ("user", """Analyze this transaction:
Amount: ${amount}
Merchant: {merchant}
Time: {time}
Customer avg spend: ${avg_spend}
Location: {location}""")
])

# Build chain with JSON parser
json_chain = json_template | llm | JsonOutputParser()

# Invoke chain
result = json_chain.invoke({
    "amount": "6800",
    "merchant": "Western Union",
    "time": "1:45 AM",
    "avg_spend": "95",
    "location": "Unknown"
})

print("=== Parsed JSON Output ===")
print(f"Type: {type(result)}")
print(f"\nRisk Level:    {result['risk_level']}")
print(f"Probability:   {result['fraud_probability']}")
print(f"Red Flags:     {result['red_flags']}")
print(f"Action:        {result['recommended_action']}")
print(f"Human Review:  {result['requires_human_review']}")

=== Parsed JSON Output ===
Type: <class 'dict'>

Risk Level:    HIGH
Probability:   0.85
Red Flags:     ['high_amount', 'late_night_transaction', 'unknown_location', 'high_merchant_risk']
Action:        Flag transaction for further review
Human Review:  True


In [13]:
# ============================================================
# CELL 5: Sequential Chain
# Multiple LLM calls where output of one feeds the next
# ============================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- Step 1: Risk Classification ---
classification_template = ChatPromptTemplate.from_messages([
    ("system", """You are a fraud risk classifier.
Classify the transaction risk in exactly this format:
RISK: HIGH/MEDIUM/LOW
AMOUNT_DEVIATION: [how many times above average]
PRIMARY_FLAG: [single most suspicious factor]"""),
    ("user", """Transaction_ID: {transaction_id},
Amount: ${amount} | Merchant: {merchant} | 
Time: {time} | Avg Spend: ${avg_spend} | 
Location: {location}""")
])

# --- Step 2: Investigation Report ---
investigation_template = ChatPromptTemplate.from_messages([
    ("system", """You are a fraud investigation report writer.
Given a risk classification, write a professional 
investigation summary for the fraud team.
Keep it under 120 words. Use professional banking language."""),
    ("user", """Transaction ID: {transaction_id}
    
Risk Classification:
{classification}

Write the investigation report.""")
])

# --- Step 3: Recommended Action ---
action_template = ChatPromptTemplate.from_messages([
    ("system", """You are a fraud operations manager.
Given an investigation report, provide a specific 
action plan in exactly 3 bullet points.
Each bullet must be a concrete, actionable step."""),
    ("user", """Investigation Report:
{investigation}

Provide 3 specific action steps.""")
])

# Build individual chains
classification_chain = classification_template | llm | StrOutputParser()
investigation_chain = investigation_template | llm | StrOutputParser()
action_chain = action_template | llm | StrOutputParser()

# --- Run the sequential pipeline ---
transaction = {
    "transaction_id": "TXN-2024-FR-9921",
    "amount": "7200",
    "merchant": "Cryptocurrency Exchange",
    "time": "3:22 AM",
    "avg_spend": "110",
    "location": "Eastern Europe"
}

print("=" * 55)
print("FRAUD INVESTIGATION PIPELINE")
print("=" * 55)

# Step 1
print("\n[STEP 1] Risk Classification...")
classification = classification_chain.invoke(transaction)
print(classification)

# Step 2 - passes classification output into next chain
print("\n[STEP 2] Generating Investigation Report...")
investigation = investigation_chain.invoke({
    "transaction_id": transaction["transaction_id"],
    "classification": classification
})
print(investigation)

# Step 3 - passes investigation into final chain
print("\n[STEP 3] Determining Action Plan...")
actions = action_chain.invoke({
    "investigation": investigation
})
print(actions)

print("\n" + "=" * 55)
print("PIPELINE COMPLETE")
print("=" * 55)

FRAUD INVESTIGATION PIPELINE

[STEP 1] Risk Classification...
RISK: HIGH
AMOUNT_DEVIATION: 65.45 (65.45 times above average)
PRIMARY_FLAG: High amount spent on a Cryptocurrency Exchange, especially at an unusual hour (3:22 AM)

[STEP 2] Generating Investigation Report...
**Investigation Summary: Transaction ID TXN-2024-FR-9921**

**Date:** [Insert Date]
**Risk Classification:** HIGH

A high-risk transaction has been identified, warranting further investigation. The transaction, TXN-2024-FR-9921, exhibits significant deviation from average spending patterns, with an amount deviation of 65.45 times above average. Notably, the transaction occurred at 3:22 AM, an unusual hour for financial activity. Furthermore, the primary flag indicates a high amount spent on a Cryptocurrency Exchange, suggesting potential unauthorized or suspicious activity. Further analysis is required to determine the legitimacy of this transaction and prevent potential financial losses.

[STEP 3] Determining Action P

### Summary — LangChain Fundamentals

**Components learned:**
- ChatGroq — LangChain's Groq integration
- ChatPromptTemplate — reusable parameterized prompts
- StrOutputParser — clean string extraction
- JsonOutputParser — automatic JSON parsing
- LCEL pipe operator (|) — connecting components
- Sequential chains — multi-step LLM pipelines

**Project 3: Multi-step Fraud Investigation Pipeline**
Three coordinated LLM calls that classify risk, write an 
investigation report, and generate an action plan from 
a single transaction input.

Business value: Automates fraud investigation documentation.
A fraud analyst receives a complete investigation package 
instead of raw transaction data — reducing triage time 
from minutes to seconds.